# 02 - SIIM-ISIC 2020 Binary Training Extension

Purpose: add a patient-level binary melanoma training/evaluation scaffold for SIIM-ISIC 2020. This notebook is separate from RQ6 external validation and does not modify existing notebooks.

This notebook is optional. It will not run until SIIM-ISIC 2020 data is downloaded locally.

Expected local data layout:

```text
Skin_Lesion_XAI_research/data/external/siim_isic_2020/
  train-metadata.csv
  train-image/
    ISIC_0000000.jpg
```

One compatible source is Kaggle dataset `nischaydnk/isic-2020-jpg-224x224-resized`, which contains `train-metadata.csv` and `train-image/`. Keep this data local and do not commit it.

The key additions are `patient_id` grouped splits, `WeightedRandomSampler`, `BCEWithLogitsLoss`, `AdamW`, `OneCycleLR`, and ROC-AUC.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
RESEARCH_DIR = NOTEBOOK_DIR.parent
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

LOCAL_DATA_ROOT = RESEARCH_DIR / "data" / "external" / "siim_isic_2020"
DATASET_SLUG = "nischaydnk/isic-2020-jpg-224x224-resized"

def find_siim_layout(root: Path):
    """Return the folder that contains train-metadata.csv and train-image/."""
    candidates = [root] + [p.parent for p in root.rglob("train-metadata.csv")]
    for candidate in candidates:
        if (candidate / "train-metadata.csv").exists() and (candidate / "train-image").exists():
            return candidate
    return None

DATA_ROOT = find_siim_layout(LOCAL_DATA_ROOT)
if DATA_ROOT is None:
    print(f"Local SIIM data not found under {LOCAL_DATA_ROOT}")
    print(f"Downloading via kagglehub: {DATASET_SLUG}")
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "kagglehub is required for automatic SIIM-ISIC download. Install it in this notebook kernel with: pip install kagglehub"
        ) from exc
    downloaded_root = Path(kagglehub.dataset_download(DATASET_SLUG))
    DATA_ROOT = find_siim_layout(downloaded_root)
    if DATA_ROOT is None:
        raise FileNotFoundError(
            f"Downloaded {DATASET_SLUG} to {downloaded_root}, but could not find train-metadata.csv and train-image/."
        )

CSV_PATH = DATA_ROOT / "train-metadata.csv"
IMAGE_DIR = DATA_ROOT / "train-image"
IMAGE_DIR_CANDIDATES = [IMAGE_DIR, IMAGE_DIR / "image", DATA_ROOT / "image"]
IMAGE_DIR = next((path for path in IMAGE_DIR_CANDIDATES if path.exists() and any(path.glob("*.jpg"))), IMAGE_DIR)
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
MODEL_DIR = RESEARCH_DIR / "outputs" / "models"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("data root:", DATA_ROOT)
print("metadata:", CSV_PATH)
print("images:", IMAGE_DIR)
print("exists:", CSV_PATH.exists(), IMAGE_DIR.exists())

Local SIIM data not found under c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\data\external\siim_isic_2020


c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\skin-lesion-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data root: C:\Users\saiyu\.cache\kagglehub\datasets\nischaydnk\isic-2020-jpg-224x224-resized\versions\1
metadata: C:\Users\saiyu\.cache\kagglehub\datasets\nischaydnk\isic-2020-jpg-224x224-resized\versions\1\train-metadata.csv
images: C:\Users\saiyu\.cache\kagglehub\datasets\nischaydnk\isic-2020-jpg-224x224-resized\versions\1\train-image\image
exists: True True


In [2]:
import pandas as pd
from pathlib import Path

if not CSV_PATH.exists() or not IMAGE_DIR.exists():
    raise FileNotFoundError(
        "SIIM-ISIC 2020 data is not available after the setup/download cell.\n\n"
        f"Expected metadata: {CSV_PATH}\n"
        f"Expected images:   {IMAGE_DIR}\n\n"
        "Run the previous cell again after installing kagglehub, or manually place train-metadata.csv and train-image/ under "
        f"{LOCAL_DATA_ROOT}."
    )

df = pd.read_csv(CSV_PATH)
required = {"isic_id", "patient_id", "target"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

def resolve_image(isic_id):
    for image_dir in IMAGE_DIR_CANDIDATES:
        for suffix in (".jpg", ".jpeg", ".png"):
            p = image_dir / f"{isic_id}{suffix}"
            if p.exists():
                return str(p)
    return None

df["image_path"] = df["isic_id"].map(resolve_image)
before_image_filter = len(df)
df = df.dropna(subset=["image_path", "patient_id", "target"]).reset_index(drop=True)
if df.empty:
    raise FileNotFoundError(
        f"Loaded {CSV_PATH}, but none of the {before_image_filter} images were found in {IMAGE_DIR}. "
        "Check that train-image/ contains files named like ISIC_0000000.jpg."
    )
df["target"] = df["target"].astype(int)
print("usable rows:", len(df))
print("positive rate:", df.target.mean().round(4))
print("patients:", df.patient_id.nunique())

usable rows: 33126
positive rate: 0.0176
patients: 2056


In [3]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(df, df["target"], groups=df["patient_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

overlap = set(train_df.patient_id).intersection(set(val_df.patient_id))
print("train:", len(train_df), "val:", len(val_df))
print("train positive rate:", train_df.target.mean().round(4))
print("val positive rate:", val_df.target.mean().round(4))
print("patient overlap:", len(overlap))

train: 26161 val: 6965
train positive rate: 0.0175
val positive rate: 0.0182
patient overlap: 0


In [4]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 3  # increase for full experiment
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.70, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SIIMDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row.image_path).convert("RGB")
        return self.transform(image), torch.tensor(row.target, dtype=torch.float32)

class_counts = train_df.target.value_counts().sort_index()
weights = train_df.target.map(lambda y: 1.0 / class_counts.loc[y]).to_numpy()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
train_loader = DataLoader(SIIMDataset(train_df, train_tfms), batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(SIIMDataset(val_df, val_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)
print("device:", DEVICE)

device: cuda


In [5]:
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, epochs=EPOCHS, steps_per_epoch=len(train_loader))
criterion = nn.BCEWithLogitsLoss()

def evaluate(loader):
    model.eval()
    probs, labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x).squeeze(1)
            loss = criterion(logits, y)
            total_loss += loss.item() * len(y)
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    pred = [int(p >= 0.5) for p in probs]
    return {
        "loss": total_loss / len(loader.dataset),
        "roc_auc": roc_auc_score(labels, probs),
        "accuracy": accuracy_score(labels, pred),
        "balanced_accuracy": balanced_accuracy_score(labels, pred),
    }

history = []
best_auc = -1
best_path = MODEL_DIR / "siim_isic2020_resnet50_binary_best.pth"
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x).squeeze(1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * len(y)
    row = evaluate(val_loader)
    row.update({"epoch": epoch, "train_loss": train_loss / len(train_loader.dataset)})
    history.append(row)
    print(row)
    if row["roc_auc"] > best_auc:
        best_auc = row["roc_auc"]
        torch.save({"model_state_dict": model.state_dict(), "metrics": row}, best_path)

pd.DataFrame(history).to_csv(METRICS_DIR / "TRAINING_siim_isic2020_resnet50_binary.csv", index=False)
print("saved metrics:", METRICS_DIR / "TRAINING_siim_isic2020_resnet50_binary.csv")
print("saved best checkpoint:", best_path)

epoch 1/3: 100%|██████████| 818/818 [06:14<00:00,  2.18it/s]


{'loss': 0.17036167593714324, 'roc_auc': np.float64(0.8668291829125336), 'accuracy': 0.925197415649677, 'balanced_accuracy': np.float64(0.7223430666516203), 'epoch': 1, 'train_loss': 0.3988868797195873}


epoch 2/3: 100%|██████████| 818/818 [06:36<00:00,  2.06it/s]


{'loss': 0.17522630444851525, 'roc_auc': np.float64(0.8634264750249302), 'accuracy': 0.9381191672648959, 'balanced_accuracy': np.float64(0.7096045028591959), 'epoch': 2, 'train_loss': 0.20064125440451294}


epoch 3/3: 100%|██████████| 818/818 [05:47<00:00,  2.35it/s]


{'loss': 0.12960299610963247, 'roc_auc': np.float64(0.8490372236667257), 'accuracy': 0.9623833452979181, 'balanced_accuracy': np.float64(0.6215008532678662), 'epoch': 3, 'train_loss': 0.07750916930057875}
saved metrics: c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\notebooks\outputs\metrics\TRAINING_siim_isic2020_resnet50_binary.csv
saved best checkpoint: c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_XAI_research\outputs\models\siim_isic2020_resnet50_binary_best.pth
